# 01 - Collecte des données

À lancer **rarement** (par exemple une fois par mois) : récupère la courbe BAM, la liste des emprunts obligataires AMMC et lit les notes d'opération PDF.

| Étape | Source | Résultat (cache) | Durée la 1re fois |
|---|---|---|---|
| 1 | Bank Al-Maghrib | `data/courbe_bam.csv` | quelques secondes |
| 2 | AMMC (liste + pages de détail) | `data/ammc_emissions_details.csv` | ~10 min |
| 3 | Notes d'opération PDF | `data/pdfs/` + `data/tranches.csv` | ~5 min |

Chaque étape relit son cache s'il existe. Pour tout reprendre de zéro : `rafraichir=True`.

Ensuite, passe à **`02_analyse.ipynb`**.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)

from courbe import get_courbe_bam, taux_bam
from ammc import collecter_emissions, extraire_emetteur
from pipeline import construire_tranches

## 1. Courbe des taux BDT (Bank Al-Maghrib)

In [2]:
courbe = get_courbe_bam(rafraichir=True)      # True = courbe du jour
print("Date de valeur :", pd.to_datetime(courbe["date_valeur"]).iloc[0].date())
print("BDT 5 ans :", round(taux_bam(5, courbe) * 100, 3), "%  |  BDT 10 ans :", round(taux_bam(10, courbe) * 100, 3), "%")
courbe

Date de valeur : 2026-11-12
BDT 5 ans : 3.065 %  |  BDT 10 ans : 3.715 %


,echeance,maturite,taux,date_valeur
0,2027-01-18,0.183562,0.0217,2026-11-12
1,2027-02-15,0.260274,0.0220,2026-11-12
2,2027-08-16,0.758904,0.0229,2026-11-12
3,2028-02-14,1.257534,0.0244,2026-11-12
4,2030-04-15,3.424658,0.0283,2026-11-12
5,2032-04-19,5.438356,0.0313,2026-11-12
6,2035-06-18,8.602740,0.0353,2026-11-12
7,2039-07-18,12.687671,0.0407,2026-11-12
8,2045-08-14,18.767123,0.0435,2026-11-12
9,2055-04-19,28.452055,0.0460,2026-11-12


## 2. Emprunts obligataires visés par l'AMMC

In [3]:
base = collecter_emissions(rafraichir=False)
base["emetteur"] = base["operation"].apply(extraire_emetteur)     # noms normalisés
print(len(base), "émissions calculables |", base["pdf_note"].notna().sum(), "avec note d'opération")
base["type"].value_counts()

197 émissions calculables | 192 avec note d'opération


type
ordinaire     118
subordonné     78
garanti         1
Name: count, dtype: int64

## 3. Lecture des notes d'opération (une ligne par tranche)

In [4]:
tranches = construire_tranches(base, rafraichir=False)
print(len(tranches), "tranches extraites")
tranches.head(10)

PDF : 10/192  (0 s)
PDF : 20/192  (0 s)
PDF : 30/192  (0 s)
PDF : 40/192  (0 s)
PDF : 50/192  (0 s)
PDF : 60/192  (1 s)
PDF : 70/192  (1 s)
PDF : 80/192  (1 s)
PDF : 90/192  (1 s)
PDF : 100/192  (1 s)
PDF : 110/192  (1 s)
PDF : 120/192  (1 s)
PDF : 130/192  (1 s)
PDF : 140/192  (1 s)
PDF : 150/192  (-0 s)
PDF : 160/192  (-0 s)
PDF : 170/192  (0 s)
PDF : 180/192  (0 s)
PDF : 190/192  (1 s)
472 tranches extraites


,tranche,jouissance,echeance,maturite_ans,nature_taux,prime_pb,remboursement,differe_ans,emetteur,type,date_visa,montant_mdh,pdf_note,erreur
0,A,2025-12-29,2035-12-29,10.0,fixe,70.0,in fine,0.0,CFG BANK,subordonné,2025-12-12,500.0000,https://www.ammc.ma/sites/default/files/NO_EOS...,NaN
1,B,2025-12-29,2035-12-29,10.0,révisable,65.0,in fine,0.0,CFG BANK,subordonné,2025-12-12,500.0000,https://www.ammc.ma/sites/default/files/NO_EOS...,NaN
2,A,2025-12-19,2040-12-19,15.0,révisable,60.0,amortissable,0.0,FEC,ordinaire,2025-12-03,2000.0000,https://www.ammc.ma/sites/default/files/NO_FEC...,NaN
3,B,2025-12-19,2040-12-19,15.0,révisable,47.5,amortissable,0.0,FEC,ordinaire,2025-12-03,2000.0000,https://www.ammc.ma/sites/default/files/NO_FEC...,NaN
4,A,2025-12-17,2027-12-17,2.0,révisable,45.0,in fine,0.0,RCI FINANCE MAROC,ordinaire,2025-11-28,900.0000,https://www.ammc.ma/sites/default/files/Note_R...,NaN
5,B,2025-12-17,2028-12-17,3.0,révisable,55.0,in fine,0.0,RCI FINANCE MAROC,ordinaire,2025-11-28,900.0000,https://www.ammc.ma/sites/default/files/Note_R...,NaN
6,C,2025-12-17,2029-12-17,4.0,révisable,65.0,in fine,0.0,RCI FINANCE MAROC,ordinaire,2025-11-28,900.0000,https://www.ammc.ma/sites/default/files/Note_R...,NaN
7,A,2025-12-23,2032-12-23,7.0,révisable,130.0,amortissable,3.0,ALLIANCES DÉVELOPPEMENT IMMOBILIER,garanti,2025-11-29,449.4204,https://www.ammc.ma/sites/default/files/NO_ADI...,NaN
8,A,2025-10-27,2035-10-27,10.0,fixe,77.5,in fine,0.0,CDM,subordonné,2025-10-09,1000.0000,https://www.ammc.ma/sites/default/files/NO_CDM...,NaN
9,B,2025-10-27,2035-10-27,10.0,révisable,72.5,in fine,0.0,CDM,subordonné,2025-10-09,1000.0000,https://www.ammc.ma/sites/default/files/NO_CDM...,NaN


### Contrôle qualité : part d'informations manquantes

In [5]:
colonnes = ["echeance", "maturite_ans", "nature_taux", "prime_pb", "remboursement"]
(tranches[colonnes].isna().mean() * 100).round(1).rename("% manquant")

echeance         65.3
maturite_ans     11.9
nature_taux      12.1
prime_pb         15.7
remboursement    26.5
Name: % manquant, dtype: float64

Les données sont prêtes. Passe à **`02_analyse.ipynb`**.